# Round 3 — 전체 데이터 최종 fine-tuning
- 베이스 모델: `runs2/round2/weights/best.pt`
- 데이터: 전체 통합 데이터셋
- 저장 위치: `runs2/round3`
- MLflow: 자동 로깅

In [ ]:
import mlflow
from ultralytics import YOLO

# ── 경로 설정 ──────────────────────────────────────
DATA_YAML   = '/mnt/ml/helmet_project/data/data_final/data.yaml'
PROJECT_DIR = '/mnt/ml/helmet_project/experiments/runs2'
RUN_NAME    = 'round3'
BASE_MODEL = '/mnt/ml/helmet_project/models/pretrained/yolov8s.pt'

# ── MLflow 설정 ────────────────────────────────────
mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment('helmet_detection')

with mlflow.start_run(run_name=RUN_NAME):

    mlflow.log_params({
        'base_model'  : BASE_MODEL,
        'data'        : DATA_YAML,
        'epochs'      : 100,
        'imgsz'       : 640,
        'batch'       : 32,
        'round'       : 3,
        'cls_weight'  : 2.0,
        'note'        : 'full_data_final_finetune',
    })

    model = YOLO(BASE_MODEL)  # round2 best.pt 로드

    results = model.train(
        data          = DATA_YAML,
        epochs        = 100,
        imgsz         = 640,
        batch         = 32,
        device        = 0,
        project       = PROJECT_DIR,
        name          = RUN_NAME,
        exist_ok      = True,
        cls           = 2.0,
        lr0 = 0.01,       # fine-tuning 아니라 새 학습
        warmup_epochs = 3,
        patience      = 20,       # early stopping
    )

    metrics = results.results_dict
    mlflow.log_metrics({
        'mAP50'     : metrics.get('metrics/mAP50(B)', 0),
        'mAP50_95'  : metrics.get('metrics/mAP50-95(B)', 0),
        'precision' : metrics.get('metrics/precision(B)', 0),
        'recall'    : metrics.get('metrics/recall(B)', 0),
    })

    best_pt = f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt'
    mlflow.log_artifact(best_pt, artifact_path='weights')

    print(f'\n✅ Round3 완료 — 최종 모델')
    print(f'   best.pt: {best_pt}')
    print(f'   mAP50  : {metrics.get("metrics/mAP50(B)", 0):.4f}')

In [ ]:
# 최종 테스트셋 검증
from ultralytics import YOLO

model = YOLO('/mnt/ml/helmet_project/experiments/runs2/round3/weights/best.pt')
val   = model.val(
    data   = '/mnt/ml/helmet_project/data/data_remapped/data.yaml',
    split  = 'test',
    conf   = 0.5,
)
print(val)

In [ ]:
# MLflow에서 세 라운드 비교
import mlflow

mlflow.set_tracking_uri('http://127.0.0.1:5000')
client = mlflow.tracking.MlflowClient()
exp    = client.get_experiment_by_name('helmet_detection')
runs   = client.search_runs(
    experiment_ids = [exp.experiment_id],
    order_by       = ['metrics.mAP50 DESC'],
)

print(f'{'Run':<10} {'mAP50':>8} {'mAP50-95':>10} {'Precision':>10} {'Recall':>8}')
print('-' * 52)
for r in runs:
    m = r.data.metrics
    print(
        f"{r.data.tags.get('mlflow.runName','?'):<10}"
        f"{m.get('mAP50', 0):>8.4f}"
        f"{m.get('mAP50_95', 0):>10.4f}"
        f"{m.get('precision', 0):>10.4f}"
        f"{m.get('recall', 0):>8.4f}"
    )